## Notebook Summary

Content!

In [1]:
# Imports

import wfdb
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

%matplotlib widget

In [2]:
# Load Data

# Stream single record directly from PhysioNet: PPG and ECG peaks
record = wfdb.rdrecord('s3_walk', 
                        pn_dir='wrist/1.0.0')
annotation = wfdb.rdann('s3_walk', 
                         extension='atr',
                         pn_dir='wrist/1.0.0')

fs = 256
ppg = record.p_signal[:, 1]
accel_x = record.p_signal[:, 5]
accel_y = record.p_signal[:, 6]
accel_z = record.p_signal[:, 7]
accel_mag = np.sqrt(accel_x**2 + accel_y**2 + accel_z**2)
r_peaks = annotation.sample

t = np.arange(len(ppg)) / fs

In [12]:
# Pre-processing

# Combine accel channels
accel = accel_x + accel_y + accel_z

# Bandpass filter: 0.5 to 5 Hz (impulse and freq response checked for stability and performance)
order = 4
sos = signal.butter(order, [0.5, 5.0], btype='band', fs=fs, output='sos') 

# Apply BPF to both ppg and accel data
ppg_filt = signal.sosfilt(sos, ppg)
accel_filt = signal.sosfilt(sos, accel)

# Normalize: PPG and accel data in different units on different scales
ppg_norm = (ppg_filt - np.mean(ppg_filt)) / np.std(ppg_filt)
accel_norm = (accel_filt - np.mean(accel_filt)) / np.std(accel_filt)